In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pygame # For rendering and image handling
import time

# Ensure the project root is in sys.path for imports like 'from causalgym.envs ...'
# Get the directory of the current notebook.
# __file__ is not defined in interactive notebook environments, so use os.getcwd() as a fallback.
try:
    current_notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_notebook_dir = os.getcwd()

# Navigate up two levels from "causal2/causalgym/test/" to "causal2/"
project_root = os.path.abspath(os.path.join(current_notebook_dir, '..', '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added project root to sys.path: {project_root}")

try:
    from causalgym.envs import LunarLanderSCM
    # LunarLanderPCH was removed as it's not used in this notebook
    print("Successfully imported LunarLanderSCM.")
except ImportError as e:
    print(f"Error importing causalgym.envs.LunarLanderSCM: {e}")
    print(f"Current sys.path: {sys.path}")
    print("Please ensure 'causalgym' is a package in the project root and the project root is correctly added to sys.path.")

# Global variables (if any) can be defined here
# Example:
# DEFAULT_MAX_STEPS = 500


In [ ]:
def create_lander_env(render_mode='rgb_array', wind_mean=0.0, wind_std=0.2, max_steps=1000):
    """Helper function to create a LunarLanderSCM environment."""
    try:
        env = LunarLanderSCM(
            render_mode=render_mode, 
            wind_mean=wind_mean, 
            wind_std=wind_std,
            max_episode_steps=max_steps
        )
        print(f"LunarLanderSCM created with wind_mean={wind_mean}, wind_std={wind_std}, render_mode='{render_mode}'")
        return env
    except Exception as e:
        print(f"Error creating LunarLanderSCM: {e}")
        return None

def run_episode(env, policy=None, max_steps=500, render_frames=True, print_wind=True):
    """Runs a single episode in the environment and optionally renders it."""
    if env is None:
        print("Environment is None, cannot run episode.")
        return [], 0, False

    frames = []
    try:
        obs, info = env.reset()
        if print_wind and hasattr(env, 'current_wind') and env.current_wind is not None:
            print(f"Episode Wind: {env.current_wind:.2f}")
        else:
            if print_wind:
                print("Could not retrieve current_wind from environment.")

    except Exception as e:
        print(f"Error during env.reset(): {e}")
        return [], 0, False
        
    total_reward = 0
    terminated = False
    truncated = False
    
    for step_count in range(max_steps):
        if render_frames:
            try:
                frame = env.render()
                if frame is not None:
                    frames.append(frame)
                else:
                    # If render_mode is human, frame might be None but rendering happens in a window
                    if env._env.render_mode == 'human':
                        pygame.display.flip() # Ensure display updates
                        time.sleep(0.03) # Small delay for human viewing
                    else:
                        print(f"Warning: env.render() returned None at step {step_count} for mode {env._env.render_mode}")
            except Exception as e:
                print(f"Error during env.render() at step {step_count}: {e}")
                break # Stop if rendering fails

        if policy is None:
            action = env.action_space.sample() # Random action
        else:
            action = policy(obs) # Policy-based action
        
        try:
            obs, reward, terminated, truncated, info = env.step(action)
        except Exception as e:
            print(f"Error during env.step() at step {step_count}: {e}")
            break # Stop if step fails
            
        total_reward += reward
        if terminated or truncated:
            break
            
    print(f"Episode finished after {step_count + 1} steps. Total reward: {total_reward:.2f}")
    if terminated:
        print("Lander landed successfully or crashed.")
    if truncated:
        print("Episode truncated (max steps reached).")
        
    return frames, total_reward, terminated

def display_frames(frames, figsize=(10, 8)):
    """Displays a list of frames (RGB arrays) as an animation or sequence."""
    if not frames:
        print("No frames to display.")
        return
    
    # Check if frames are Pygame surfaces or numpy arrays
    if isinstance(frames[0], np.ndarray):
        # Simple display for a few frames, or animate if many
        if len(frames) <= 5:
            fig, axes = plt.subplots(1, len(frames), figsize=figsize)
            if len(frames) == 1:
                axes = [axes] # make it iterable
            for i, frame in enumerate(frames):
                axes[i].imshow(frame)
                axes[i].axis('off')
            plt.tight_layout()
            plt.show()
        else: # Animation for many frames
            fig = plt.figure(figsize=figsize)
            plt.axis('off')
            img_display = plt.imshow(frames[0])
            def animate(i):
                img_display.set_data(frames[i])
                return [img_display]
            
            from matplotlib.animation import FuncAnimation
            anim = FuncAnimation(fig, animate, frames=len(frames), interval=50, blit=True)
            from IPython.display import HTML
            plt.close(fig) # close the static plot
            display(HTML(anim.to_jshtml()))
            print("Displaying animation.")

    elif hasattr(frames[0], 'get_width'): # Heuristic for Pygame surface (not perfectly robust)
        print("Frames seem to be Pygame surfaces. Consider converting to numpy arrays for display in notebook or ensure human mode rendering is active.")
    else:
        print("Unknown frame format.")

print("Helper functions defined.")

In [ ]:
# Cell 3: Test - Standard Lunar Lander (No Wind)
print("--- Test: Standard Lunar Lander (No Wind) ---")

# Create environment with no wind
# Using rgb_array for capturing frames
env_no_wind = create_lander_env(render_mode='rgb_array', wind_mean=0.0, wind_std=0.0, max_steps=300)

if env_no_wind:
    # Run an episode with random actions
    print("Running episode with random policy...")
    frames_no_wind, reward_no_wind, landed_no_wind = run_episode(env_no_wind, policy=None, max_steps=300, render_frames=True)
    
    # Display the frames
    if frames_no_wind:
        print("Displaying episode frames...")
        display_frames(frames_no_wind, figsize=(12,9))
    else:
        print("No frames were captured for the no-wind scenario.")
    
    env_no_wind.close()
    print("No-wind environment closed.")
else:
    print("Failed to create no-wind environment.")

print("--- End Test: Standard Lunar Lander (No Wind) ---")

In [ ]:
# Cell 4: Test - Lunar Lander with Positive Wind (pushes lander Right)
print("--- Test: Lunar Lander with Positive Wind (Wind from Left) ---")

# Define wind parameters - positive mean for wind from the left
WIND_STRENGTH_POSITIVE = 15.0 # Stronger wind for clear effect
WIND_STD_DEV = 2.0

# Create environment with positive wind
env_positive_wind = create_lander_env(
    render_mode='rgb_array', 
    wind_mean=WIND_STRENGTH_POSITIVE, 
    wind_std=WIND_STD_DEV, 
    max_steps=300
)

if env_positive_wind:
    print(f"Running episode with positive wind (mean={WIND_STRENGTH_POSITIVE})...")
    frames_positive_wind, reward_positive_wind, landed_positive_wind = run_episode(
        env_positive_wind, 
        policy=None, 
        max_steps=300, 
        render_frames=True
    )
    
    if frames_positive_wind:
        print("Displaying episode frames with positive wind...")
        display_frames(frames_positive_wind, figsize=(12,9))
    else:
        print("No frames were captured for the positive-wind scenario.")
        
    env_positive_wind.close()
    print("Positive-wind environment closed.")
else:
    print("Failed to create positive-wind environment.")

print("--- End Test: Lunar Lander with Positive Wind ---")

In [ ]:
# Cell 5: Test - Lunar Lander with Negative Wind (pushes lander Left)
print("--- Test: Lunar Lander with Negative Wind (Wind from Right) ---")

# Define wind parameters - negative mean for wind from the right
WIND_STRENGTH_NEGATIVE = -15.0 # Stronger wind for clear effect
# WIND_STD_DEV is already defined or use a new one if needed, e.g., 2.0

# Create environment with negative wind
env_negative_wind = create_lander_env(
    render_mode='rgb_array', 
    wind_mean=WIND_STRENGTH_NEGATIVE, 
    wind_std=WIND_STD_DEV, # Using the same std dev as positive wind test
    max_steps=300
)

if env_negative_wind:
    print(f"Running episode with negative wind (mean={WIND_STRENGTH_NEGATIVE})...")
    frames_negative_wind, reward_negative_wind, landed_negative_wind = run_episode(
        env_negative_wind, 
        policy=None, 
        max_steps=300, 
        render_frames=True
    )
    
    if frames_negative_wind:
        print("Displaying episode frames with negative wind...")
        display_frames(frames_negative_wind, figsize=(12,9))
    else:
        print("No frames were captured for the negative-wind scenario.")
        
    env_negative_wind.close()
    print("Negative-wind environment closed.")
else:
    print("Failed to create negative-wind environment.")

print("--- End Test: Lunar Lander with Negative Wind ---")

In [ ]:
# Cell 6: Test - Observing Success/Failure with Moderate Wind
print("--- Test: Observing Success/Failure with Moderate Wind ---")

# Moderate wind that might allow for both success and failure with a random policy
MODERATE_WIND_MEAN = 5.0 
MODERATE_WIND_STD = 1.0
NUM_EPISODES_OBSERVE = 3 # Run a few episodes to see different outcomes

env_observe = create_lander_env(
    render_mode='rgb_array', 
    wind_mean=MODERATE_WIND_MEAN, 
    wind_std=MODERATE_WIND_STD, 
    max_steps=400 # Slightly longer episodes
)

if env_observe:
    for i in range(NUM_EPISODES_OBSERVE):
        print(f"\n--- Running Observation Episode {i+1}/{NUM_EPISODES_OBSERVE} with moderate wind (mean={MODERATE_WIND_MEAN}) ---")
        frames_observe, reward_observe, landed_observe = run_episode(
            env_observe, 
            policy=None, 
            max_steps=400, 
            render_frames=True
        )
        
        if frames_observe:
            print(f"Displaying frames for episode {i+1}...")
            display_frames(frames_observe, figsize=(12,9))
            if landed_observe:
                print(f"Outcome for episode {i+1}: Successful landing (or crash, reward: {reward_observe:.2f})")
            else:
                print(f"Outcome for episode {i+1}: Episode truncated (reward: {reward_observe:.2f})")
        else:
            print(f"No frames captured for observation episode {i+1}.")
        print("-----------------------------------------------------")
            
    env_observe.close()
    print("Observation environment closed.")
else:
    print("Failed to create environment for observing success/failure.")

print("--- End Test: Observing Success/Failure with Moderate Wind ---")

In [ ]:
# Cell 7: Cleanup
# Though environments are closed in each test cell, this is a placeholder for any global cleanup if needed.

# For example, if Pygame was initialized globally:
# pygame.quit()

print("Notebook execution complete. All test environments should be closed.")
print("If you ran any cells with render_mode='human', ensure Pygame windows are closed manually if they persist.")

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
import random
import io
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage, Markdown
import os
import sys

# Minigrid specific imports
from minigrid.minigrid_env import MiniGridEnv
from minigrid.core.constants import COLOR_NAMES, OBJECT_TO_IDX, IDX_TO_OBJECT
from minigrid.core.grid import Grid
from minigrid.core.mission import MissionSpace
from minigrid.core.world_object import Door, Goal, Key, Lava, Wall, Ball, Box
from minigrid.wrappers import ImgObsWrapper, RGBImgPartialObsWrapper, ReseedWrapper

# Path adjustments for causalgym
# Assuming notebook is in causal2/test/
# and causalgym module is in causal2/causalgym/
# Get current notebook directory
notebook_dir = os.path.dirname(os.path.abspath("__file__")) # oslint-disable-line # __file__ is not defined in notebook
if not notebook_dir or "__file__" in notebook_dir: # Fallback for interactive sessions
    notebook_dir = os.getcwd()

# Construct path to 'causal2' directory (workspace root)
workspace_root = os.path.abspath(os.path.join(notebook_dir, '..')) # Assumes notebook is in causal2/test

# Add workspace root to sys.path to allow imports like `from causalgym.causal_gym...`
if workspace_root not in sys.path:
    sys.path.append(workspace_root)

print(f"Workspace root (added to sys.path): {workspace_root}")
print(f"Current sys.path: {sys.path}")

# Try importing from causal_gym
try:
    # These imports are based on your typical project structure.
    # Adjust if your causalgym module is structured differently.
    from causalgym.causal_gym import envs  # Example: To register custom envs if any
    from causalgym.causal_gym import wrappers # Example: If there are generic wrappers
    print("Successfully imported from causalgym (or modules exist).")
except ImportError as e:
    print(f"Could not import from causalgym.causal_gym, ensure causalgym is in sys.path and structured correctly: {e}")
except ModuleNotFoundError as e:
    print(f"ModuleNotFoundError when importing from causalgym.causal_gym: {e}")


print("Imports completed.")

In [ ]:
# Set seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# For gymnasium environments, seed is usually passed to make() or reset()

def show_obs(obs, title="Observation", save_path=None, enlarge=False):
    img_array = None
    if isinstance(obs, dict) and "image" in obs: # Minigrid often uses 'image' key for agent view
         img_array = obs["image"]
    elif isinstance(obs, dict) and "rgb_array" in obs: # For SCM-based envs like our FrozenLake
        img_array = obs["rgb_array"]
    elif isinstance(obs, np.ndarray) and obs.ndim == 3 and obs.shape[2] in [1, 3, 4]: # Typical gym envs returning rgb_array or similar
        img_array = obs
    else:
        print(f"Unsupported observation format for show_obs: {type(obs)}")
        if isinstance(obs, dict): print(f"Keys: {obs.keys()}")
        return

    if img_array is None:
        print("Could not extract image array from observation.")
        return
        
    # Enlarge if requested (useful for small Minigrid images)
    if enlarge and img_array.shape[0] < 100 and img_array.shape[1] < 100 : # Heuristic for small images
        target_width = max(256, img_array.shape[1] * 5) # Ensure reasonable enlargement
        scale = target_width / img_array.shape[1]
        target_height = int(img_array.shape[0] * scale)
        try:
            pil_img = PILImage.fromarray(img_array.astype(np.uint8))
            pil_img = pil_img.resize((target_width, target_height), PILImage.NEAREST)
            img_array_to_show = np.array(pil_img)
        except Exception as e:
            print(f"Error during image resize: {e}")
            img_array_to_show = img_array # Show original if resize fails
    else:
        img_array_to_show = img_array

    plt.imshow(img_array_to_show)
    plt.title(title)
    plt.axis('off')
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
        print(f"Observation saved to {save_path}")
    plt.show()

# Placeholder WindyWrapper
class WindyWrapper(gym.Wrapper):
    def __init__(self, env, wind_config=None):
        super().__init__(env)
        self.np_random_wind = np.random.RandomState()
        self.np_random_wind.seed(SEED) # Separate RNG for wind, seeded

        self.wind_type = wind_config.get('type', 'global') if wind_config else 'global'
        self.wind_strength = wind_config.get('strength', 1) if wind_config else 1
        
        self.global_wind_direction = None # 0: R (East), 1: D (South), 2: L (West), 3: U (North)
        self.wind_map = None # (rows, cols) -> direction (0-3, or 4 for no wind)

        self.grid_height = env.height
        self.grid_width = env.width
        
        self.current_agent_pos = env.agent_pos
        self.current_agent_dir = env.agent_dir
        self._sample_wind()

    def seed(self, seed=None):
        super().seed(seed) # Seeds the underlying env's RNGs
        if seed is not None:
            self.np_random_wind.seed(seed) # Reseed wrapper's wind RNG as well

    def _sample_wind(self):
        if self.wind_type == 'global':
            self.global_wind_direction = self.np_random_wind.integers(0, 4)
            self.wind_map = None
        elif self.wind_type == 'per_cell':
            self.wind_map = self.np_random_wind.integers(0, 5, size=(self.grid_height, self.grid_width))
            self.global_wind_direction = None

    def reset(self, **kwargs):
        seed = kwargs.get('seed', None)
        if seed is not None: # Important to seed the wrapper for its own RNG if a seed is passed to reset
            self.np_random_wind.seed(seed)
        
        obs, info = self.env.reset(**kwargs)
        self.current_agent_pos = self.env.agent_pos
        self.current_agent_dir = self.env.agent_dir
        self._sample_wind() # Re-sample wind on reset
        
        info['wind_type'] = self.wind_type
        if self.wind_type == 'global':
            info['global_wind_direction'] = self.global_wind_direction
        elif self.wind_type == 'per_cell' and self.wind_map is not None and self.current_agent_pos:
            r, c = self.current_agent_pos[1], self.current_agent_pos[0] # Minigrid pos is (x,y) or (col,row)
            info['current_cell_wind'] = self.wind_map[r, c] if 0 <= r < self.grid_height and 0 <= c < self.grid_width else 'N/A'
        return obs, info

    def step(self, action):
        # Actual wind effect on agent movement is NOT implemented here.
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.current_agent_pos = self.env.agent_pos 
        self.current_agent_dir = self.env.agent_dir

        info['wind_type'] = self.wind_type
        if self.wind_type == 'global':
            info['global_wind_direction'] = self.global_wind_direction
        elif self.wind_type == 'per_cell' and self.wind_map is not None and self.current_agent_pos:
            r, c = self.current_agent_pos[1], self.current_agent_pos[0] # Minigrid pos (col,row)
            if 0 <= r < self.grid_height and 0 <= c < self.grid_width:
                info['current_cell_wind'] = self.wind_map[r, c]
        return obs, reward, terminated, truncated, info

    def manually_set_agent_pos(self, pos_tuple, direction=None):
        self.env.agent_pos = pos_tuple
        self.current_agent_pos = pos_tuple
        if direction is not None:
            self.env.agent_dir = direction
            self.current_agent_dir = direction
        print(f"Agent pos set to: {self.env.agent_pos}, dir: {self.env.agent_dir}")

    def manually_set_wind(self, wind_data, wind_type='global'):
        self.wind_type = wind_type
        if wind_type == 'global':
            self.global_wind_direction = wind_data
            self.wind_map = None
            print(f"Global wind set to: {self.global_wind_direction}")
        elif wind_type == 'per_cell':
            if not isinstance(wind_data, np.ndarray) or wind_data.shape != (self.grid_height, self.grid_width):
                print(f"Error: wind_data for per_cell needs shape ({self.grid_height}, {self.grid_width})")
                return
            self.wind_map = wind_data
            self.global_wind_direction = None
            print("Per-cell wind map set.")
            
    def get_current_wind_info(self):
        info_dict = {'wind_type': self.wind_type}
        if self.wind_type == 'global':
            info_dict['global_wind_direction'] = self.global_wind_direction
        elif self.wind_type == 'per_cell' and self.wind_map is not None:
            info_dict['wind_map_shape'] = self.wind_map.shape
            if self.current_agent_pos:
                r, c = self.current_agent_pos[1], self.current_agent_pos[0]
                if 0 <= r < self.grid_height and 0 <= c < self.grid_width:
                    info_dict['current_cell_wind'] = self.wind_map[r,c]
        return info_dict

def get_minigrid_rgb_image(env_or_wrapper):
    # Standard way for gymnasium to get full RGB array render
    try:
        return env_or_wrapper.render()
    except Exception as e1:
        print(f"env.render() failed: {e1}")
        # Fallback for older or differently structured minigrid envs
        if hasattr(env_or_wrapper.unwrapped, 'grid') and hasattr(env_or_wrapper.unwrapped.grid, 'render'):
            try:
                tile_size = getattr(env_or_wrapper.unwrapped, 'tile_size', 8) # Minigrid default
                img = env_or_wrapper.unwrapped.grid.render(
                    tile_size,
                    env_or_wrapper.unwrapped.agent_pos,
                    env_or_wrapper.unwrapped.agent_dir,
                )
                return img
            except Exception as e2:
                print(f"grid.render() failed: {e2}") 
        print("Could not get RGB image from this env for custom rendering.")
        return None

def show_map_with_arrows(env_wrapper, data_map, arrow_type='wind', title="Map with Arrows"):
    print(f"--- Placeholder: show_map_with_arrows for {title} ({arrow_type}) ---")
    base_img = get_minigrid_rgb_image(env_wrapper)
    if base_img is not None:
        # Actual arrow drawing would happen here on a copy of base_img
        # For now, just show the base image and print data map info.
        show_obs(base_img, title=f"{title} (Base Env - Arrows NOT Drawn)", enlarge=True)
        if isinstance(data_map, np.ndarray):
            print(f"Data map for arrows (shape {data_map.shape}):\n{data_map[:5,:5]}... (showing top-left 5x5)")
        else:
            print(f"Data for arrows: {data_map}")
    else:
        print("Could not render base environment for arrow map.")

print("Helper functions, WindyWrapper placeholder, and show_map_with_arrows placeholder defined.")

# MiniGrid Empty World Tests

In [ ]:
# Create MiniGrid-Empty-8x8-v0
# Standard MiniGrid environments return a dict obs by default.
# render_mode='rgb_array' makes env.render() return the full grid image.
env_empty_raw = gym.make("MiniGrid-Empty-8x8-v0", render_mode="rgb_array")

# The SEED variable is global. For gym.make, seed is passed at reset or make.
obs_empty_raw, info_empty_raw = env_empty_raw.reset(seed=SEED)

print(f"Raw Empty Env observation (agent-centric view) keys: {obs_empty_raw.keys()}")
show_obs(obs_empty_raw, title="MiniGrid-Empty-8x8-v0 (Agent View)", enlarge=True)

# To show the full grid, we use env.render()
full_grid_img_empty = get_minigrid_rgb_image(env_empty_raw)
if full_grid_img_empty is not None:
    show_obs(full_grid_img_empty, title="MiniGrid-Empty-8x8-v0 (Full Grid Render)", enlarge=True)

# Agent position and direction after reset
print(f"Agent position: {env_empty_raw.agent_pos}, Agent direction: {env_empty_raw.agent_dir}")

In [ ]:
# Wrap with wind (using placeholder WindyWrapper with global wind)
env_empty_windy = WindyWrapper(env_empty_raw, wind_config={'type': 'global'})
# Seed the wrapper to ensure its internal RNG (for wind sampling) is also seeded.
# The underlying env (env_empty_raw) was already seeded at its reset.
# For full reproducibility, reset the wrapped env with a seed.
obs_empty_windy, info_empty_windy = env_empty_windy.reset(seed=SEED)

print("Empty Env with Global Wind:")
print(f"Info from reset: {info_empty_windy}")

# Show agent-centric observation from wrapped env
show_obs(obs_empty_windy, title="Empty-8x8 with Global Wind (Agent View)", enlarge=True)

# Show full grid render from wrapped env
full_grid_img_empty_windy = get_minigrid_rgb_image(env_empty_windy)
if full_grid_img_empty_windy is not None:
    show_obs(full_grid_img_empty_windy, title="Empty-8x8 with Global Wind (Full Grid Render)", enlarge=True)

print(f"Agent position: {env_empty_windy.current_agent_pos}, Agent direction: {env_empty_windy.current_agent_dir}")
print(f"Wind info: {env_empty_windy.get_current_wind_info()}")

In [ ]:
# Take action with wind
# Minigrid actions: 0:left, 1:right, 2:forward, 3:pickup, 4:drop, 5:toggle, 6:done
# Agent starts facing East (dir=0) in Empty-8x8. Action 2 (forward) should move it right.
action = 2 # Move forward
print(f"Taking action: {action} (Forward)")
obs_after_action, reward, terminated, truncated, info_after_action = env_empty_windy.step(action)

print("\nAfter action (Forward) in Empty Env with Global Wind:")
print(f"Reward: {reward}, Terminated: {terminated}, Truncated: {truncated}")
print(f"Info: {info_after_action}")

# Show agent-centric observation
show_obs(obs_after_action, title="Empty-8x8 with Wind (After Action - Agent View)", enlarge=True)

# Show full grid render
full_grid_img_after_action = get_minigrid_rgb_image(env_empty_windy)
if full_grid_img_after_action is not None:
    show_obs(full_grid_img_after_action, title="Empty-8x8 with Wind (After Action - Full Grid)", enlarge=True)

print(f"Agent position: {env_empty_windy.current_agent_pos}, Agent direction: {env_empty_windy.current_agent_dir}")
print(f"Wind info: {env_empty_windy.get_current_wind_info()}")

In [ ]:
print("\nManually setting agent and wind positions in Empty-8x8 with Wind:")

# Reset first to ensure a clean state before manual override if desired, though manual_set will override.
# This reset will also resample wind according to the wrapper's current config.
obs_before_manual, info_before_manual = env_empty_windy.reset(seed=SEED) 
print(f"Wind info after reset, before manual set: {info_before_manual}")

# Manually set agent position (e.g., to (3,3), agent_dir=0 (right))
# Minigrid agent_pos is (x, y) where x is column, y is row.
# For an 8x8 grid, coords are (0-7, 0-7). Default Empty-8x8 start is (1,1).
new_agent_pos = (3, 3)
ew_agent_dir = 0 # 0: Right (East)
env_empty_windy.manually_set_agent_pos(new_agent_pos, direction=new_agent_dir)

# Manually set global wind (e.g., global wind direction to 1: Down)
new_global_wind_dir = 1 # 1: Down (South)
env_empty_windy.manually_set_wind(new_global_wind_dir, wind_type='global')

print(f"After manual settings - Agent at: {env_empty_windy.current_agent_pos}, dir: {env_empty_windy.current_agent_dir}")
print(f"After manual settings - Wind info: {env_empty_windy.get_current_wind_info()}")

# Show agent-centric observation after manual setting.
# For Minigrid, env.unwrapped.gen_obs() generates the agent's view based on current state.
obs_manual_agent_view = env_empty_windy.unwrapped.gen_obs()
show_obs(obs_manual_agent_view, title="Empty-8x8 - Manual Set (Agent View)", enlarge=True)

# Show full grid render after manual setting
full_grid_img_manual = get_minigrid_rgb_image(env_empty_windy)
if full_grid_img_manual is not None:
    show_obs(full_grid_img_manual, title="Empty-8x8 - Manual Set (Full Grid)", enlarge=True)

# MiniGrid LavaCrossing Tests

These tests will attempt to load `Custom-LavaCrossing-*` environments. If they are not registered, standard MiniGrid `LavaCrossingS*N*` environments will be used as fallbacks.

In [ ]:
# Test Custom-LavaCrossing-easy-v0
env_name_lava_easy = "Custom-LavaCrossing-easy-v0"
fallback_lava_easy = "MiniGrid-LavaCrossingS9N1-v0" # S9N1 is relatively easy
env_lava_easy_raw = None

try:
    print(f"Attempting to load: {env_name_lava_easy}")
    env_lava_easy_raw = gym.make(env_name_lava_easy, render_mode="rgb_array")
except gym.error.NameNotFound:
    print(f"ERROR: Environment '{env_name_lava_easy}' not found. Using fallback: '{fallback_lava_easy}'.")
    try:
        env_lava_easy_raw = gym.make(fallback_lava_easy, render_mode="rgb_array")
        env_name_lava_easy = fallback_lava_easy # Update name for titles
    except gym.error.NameNotFound as e:
        print(f"ERROR: Fallback environment '{fallback_lava_easy}' also not found: {e}")
        # Cannot proceed with this specific test block if env_lava_easy_raw is None

if env_lava_easy_raw:
    obs_lava_easy, info_lava_easy = env_lava_easy_raw.reset(seed=SEED)
    print(f"Loaded {env_name_lava_easy}. Agent pos: {env_lava_easy_raw.agent_pos}, dir: {env_lava_easy_raw.agent_dir}")

    show_obs(obs_lava_easy, title=f"{env_name_lava_easy} (Agent View)", enlarge=True)
    full_grid_lava_easy = get_minigrid_rgb_image(env_lava_easy_raw)
    if full_grid_lava_easy is not None:
        show_obs(full_grid_lava_easy, title=f"{env_name_lava_easy} (Full Grid Render)", enlarge=True)
else:
    print(f"Skipping tests for {env_name_lava_easy} as it could not be loaded.")

In [ ]:
# Wrap LavaCrossing with per-cell wind
if 'env_lava_easy_raw' in locals() and env_lava_easy_raw is not None:
    env_lava_easy_windy = WindyWrapper(env_lava_easy_raw, wind_config={'type': 'per_cell'})
    obs_lava_windy, info_lava_windy = env_lava_easy_windy.reset(seed=SEED)
    
    print(f"\n{env_name_lava_easy} with Per-Cell Wind:")
    print(f"Info from reset: {info_lava_windy}")

    show_obs(obs_lava_windy, title=f"{env_name_lava_easy} with Per-Cell Wind (Agent View)", enlarge=True)
    full_grid_img_lava_windy = get_minigrid_rgb_image(env_lava_easy_windy)
    if full_grid_img_lava_windy is not None:
        show_obs(full_grid_img_lava_windy, title=f"{env_name_lava_easy} with Per-Cell Wind (Full Grid)", enlarge=True)

    print(f"Agent pos: {env_lava_easy_windy.current_agent_pos}, dir: {env_lava_easy_windy.current_agent_dir}")
    current_wind_details = env_lava_easy_windy.get_current_wind_info()
    print(f"Wind info: {current_wind_details}")

    # Visualize per-cell wind map (placeholder function)
    if current_wind_details.get('wind_type') == 'per_cell' and env_lava_easy_windy.wind_map is not None:
        show_map_with_arrows(env_lava_easy_windy, env_lava_easy_windy.wind_map, arrow_type='wind', title=f"{env_name_lava_easy} Per-Cell Wind Map")
else:
    print(f"Skipping wrapping {env_name_lava_easy} with wind as base env was not loaded.")
    # Define env_lava_easy_windy as None to prevent errors in subsequent cells
    env_lava_easy_windy = None

In [ ]:
if 'env_lava_easy_windy' in locals() and env_lava_easy_windy is not None:
    action_lava = 2 # Move forward
    print(f"\nTaking action: {action_lava} (Forward) in {env_name_lava_easy} with Per-Cell Wind")
    obs_lava_action, reward_lava, term_lava, trunc_lava, info_lava_action = env_lava_easy_windy.step(action_lava)
    
    print(f"Reward: {reward_lava}, Terminated: {term_lava}, Truncated: {trunc_lava}")
    print(f"Info after action: {info_lava_action}")
    
    show_obs(obs_lava_action, title=f"{env_name_lava_easy} (After Action - Agent View)", enlarge=True)
    full_grid_lava_action = get_minigrid_rgb_image(env_lava_easy_windy)
    if full_grid_lava_action is not None:
        show_obs(full_grid_lava_action, title=f"{env_name_lava_easy} (After Action - Full Grid)", enlarge=True)

    print(f"Agent pos: {env_lava_easy_windy.current_agent_pos}, dir: {env_lava_easy_windy.current_agent_dir}")
    print(f"Wind info: {env_lava_easy_windy.get_current_wind_info()}")
else:
    print(f"Skipping action in {env_name_lava_easy} as wrapped env is not available.")

In [ ]:
# This cell demonstrates manual setting. The "(Commented-out code)" instruction from the prompt
# is interpreted as a note about the template, not to comment out this cell's active code.

if 'env_lava_easy_windy' in locals() and env_lava_easy_windy is not None:
    print(f"\nManually setting agent and wind for {env_name_lava_easy}:")
    
    # Reset for a clean state before manual override.
    # This will resample wind as per wrapper's config (per-cell for this env_lava_easy_windy instance).
    obs_before_manual_lava, info_before_manual_lava = env_lava_easy_windy.reset(seed=SEED)
    print(f"Wind info after reset, before manual set: {info_before_manual_lava}")

    # Manually set agent position and direction
    # For LavaCrossingS9N1 (9x9 grid), (0-8, 0-8). Default start (1,1).
    # Let's try agent at (1,4) (col 1, row 4), facing Down (dir 1).
    new_lava_agent_pos = (1, 4) 
    new_lava_agent_dir = 1 # 1: Down (South)
    env_lava_easy_windy.manually_set_agent_pos(new_lava_agent_pos, direction=new_lava_agent_dir)
    
    # Manually set per-cell wind (example: all wind blows right (0), except one cell)
    h = env_lava_easy_windy.grid_height
    w = env_lava_easy_windy.grid_width
    manual_wind_map_lava = np.full((h, w), 0, dtype=int) # All wind to the Right (0)
    if h > 2 and w > 2: # Make a specific cell different if grid is large enough
        manual_wind_map_lava[2,2] = 3 # Wind Up (3) at (row 2, col 2)
    env_lava_easy_windy.manually_set_wind(manual_wind_map_lava, wind_type='per_cell')
    
    print(f"After manual settings - Agent at: {env_lava_easy_windy.current_agent_pos}, dir: {env_lava_easy_windy.current_agent_dir}")
    print(f"After manual settings - Wind info: {env_lava_easy_windy.get_current_wind_info()}")

    obs_manual_lava_agent_view = env_lava_easy_windy.unwrapped.gen_obs()
    show_obs(obs_manual_lava_agent_view, title=f"{env_name_lava_easy} - Manual Set (Agent View)", enlarge=True)
    
    full_grid_lava_manual = get_minigrid_rgb_image(env_lava_easy_windy)
    if full_grid_lava_manual is not None:
        show_obs(full_grid_lava_manual, title=f"{env_name_lava_easy} - Manual Set (Full Grid)", enlarge=True)

    # Show the manually set wind map (placeholder function)
    show_map_with_arrows(env_lava_easy_windy, manual_wind_map_lava, arrow_type='wind', title=f"{env_name_lava_easy} Manually Set Wind Map")
else:
    print(f"Skipping manual set for {env_name_lava_easy} as wrapped env is not available.")

In [ ]:
if 'env_lava_easy_windy' in locals() and env_lava_easy_windy is not None:
    action_lava_after_manual = 0 # Turn Left
    # Agent was manually set to dir 1 (Down). Turning left makes it dir 0 (Right/East).
    print(f"\nTaking action: {action_lava_after_manual} (Turn Left) in {env_name_lava_easy} after manual set")
    obs_after_manual_action, rw_manual, tm_manual, tc_manual, inf_manual_action = env_lava_easy_windy.step(action_lava_after_manual)
    
    print(f"Reward: {rw_manual}, Terminated: {tm_manual}, Truncated: {tc_manual}")
    print(f"Info after action: {inf_manual_action}")
    
    show_obs(obs_after_manual_action, title=f"{env_name_lava_easy} (After Manual Set & Action - Agent View)", enlarge=True)
    full_grid_manual_action = get_minigrid_rgb_image(env_lava_easy_windy)
    if full_grid_manual_action is not None:
        show_obs(full_grid_manual_action, title=f"{env_name_lava_easy} (After Manual Set & Action - Full Grid)", enlarge=True)

    print(f"Agent pos: {env_lava_easy_windy.current_agent_pos}, dir: {env_lava_easy_windy.current_agent_dir}")
    print(f"Wind info: {env_lava_easy_windy.get_current_wind_info()}")
else:
    print(f"Skipping action after manual set for {env_name_lava_easy} as wrapped env is not available.")

In [ ]:
# Helper function for creating, wrapping, and testing other MiniGrid envs
def run_minigrid_env_test_sequence(env_name_custom, fallback_env_name_std, wind_config={'type': 'global'}, action_to_take=None, policy_viz_dummy=False, special_viz_requests=False):
    print(f"\n" + "-"*50)
    print(f"--- Testing Environment Sequence: {env_name_custom} ---")
    print("-"*50)
    
    current_env_name = env_name_custom
    env_raw = None
    try:
        print(f"Attempting to load: {current_env_name}")
        env_raw = gym.make(current_env_name, render_mode="rgb_array")
    except gym.error.NameNotFound:
        print(f"ERROR: '{current_env_name}' not found. Using fallback: '{fallback_env_name_std}'.")
        try:
            env_raw = gym.make(fallback_env_name_std, render_mode="rgb_array")
            current_env_name = fallback_env_name_std # Update name for titles
        except gym.error.NameNotFound as e:
            print(f"ERROR: Fallback '{fallback_env_name_std}' also not found: {e}. Skipping sequence.")
            return None
    
    if not env_raw:
        return None

    # Wrap with specified wind config
    env_wrapped = WindyWrapper(env_raw, wind_config=wind_config)
    obs_wrapped, info_wrapped = env_wrapped.reset(seed=SEED)

    print(f"Loaded & Wrapped {current_env_name}. Info from reset: {info_wrapped}")
    show_obs(obs_wrapped, title=f"{current_env_name} with Wind (Agent View)", enlarge=True)
    img_full_wrapped = get_minigrid_rgb_image(env_wrapped)
    if img_full_wrapped is not None:
        show_obs(img_full_wrapped, title=f"{current_env_name} with Wind (Full Grid)", enlarge=True)
    print(f"Wind info: {env_wrapped.get_current_wind_info()}")

    if env_wrapped.wind_type == 'per_cell' and env_wrapped.wind_map is not None:
        show_map_with_arrows(env_wrapped, env_wrapped.wind_map, arrow_type='wind', title=f"{current_env_name} Wind Map")
    elif env_wrapped.wind_type == 'global':
        h_glob, w_glob = env_wrapped.grid_height, env_wrapped.grid_width
        global_wind_map_viz = np.full((h_glob, w_glob), env_wrapped.global_wind_direction, dtype=int)
        show_map_with_arrows(env_wrapped, global_wind_map_viz, arrow_type='wind', title=f"{current_env_name} Global Wind Direction ({env_wrapped.global_wind_direction}) Map")

    if action_to_take is not None:
        action_val, action_name_str = action_to_take
        print(f"\nTaking action: {action_val} ({action_name_str}) in {current_env_name}")
        obs_action, reward, term, trunc, info_action = env_wrapped.step(action_val)
        print(f"Result: Reward={reward}, Term={term}, Trunc={trunc}, Info={info_action}")
        show_obs(obs_action, title=f"{current_env_name} (After Action - Agent View)", enlarge=True)
        img_full_action = get_minigrid_rgb_image(env_wrapped)
        if img_full_action is not None:
            show_obs(img_full_action, title=f"{current_env_name} (After Action - Full Grid)", enlarge=True)
        print(f"Wind info after action: {env_wrapped.get_current_wind_info()}")

    # Dummy policy for visualization if requested
    h_pol, w_pol = env_wrapped.grid_height, env_wrapped.grid_width
    # Ensure action_space is available and has n attribute
    num_actions = env_wrapped.action_space.n if hasattr(env_wrapped.action_space, 'n') else 7 # Default to 7 for Minigrid if not found
    dummy_policy_map = env_wrapped.np_random_wind.integers(0, num_actions, size=(h_pol, w_pol))

    if policy_viz_dummy:
        print(f"\nVisualizing dummy policy for {current_env_name}:")
        show_map_with_arrows(env_wrapped, dummy_policy_map, arrow_type='policy', title=f"{current_env_name} Dummy Policy Map")

    if special_viz_requests: # For Custom-LavaCrossing-maze-complex-v0
        print(f"\n--- Special Visualizations for {current_env_name} ---")
        
        # Show observation with policy (already done by policy_viz_dummy if true, but can repeat)
        print("Map with policy arrows (dummy policy):")
        show_map_with_arrows(env_wrapped, dummy_policy_map, arrow_type='policy', title=f"{current_env_name} Policy Arrows")

        # Show observation with policy and wind
        print("\nMap with wind arrows (re-show):")
        if env_wrapped.wind_type == 'per_cell' and env_wrapped.wind_map is not None:
            show_map_with_arrows(env_wrapped, env_wrapped.wind_map, arrow_type='wind', title=f"{current_env_name} Wind Arrows")
        elif env_wrapped.wind_type == 'global':
            global_wind_map_viz_special = np.full((h_pol, w_pol), env_wrapped.global_wind_direction, dtype=int)
            show_map_with_arrows(env_wrapped, global_wind_map_viz_special, arrow_type='wind', title=f"{current_env_name} Global Wind Visualization")
        print("Combined policy and wind arrow visualization requires an advanced 'show_map_with_arrows' function.")

        # Show map with all states mapped to 1 (non-wall cells)
        grid_obj = env_wrapped.unwrapped.grid
        map_all_navigable = np.zeros((h_pol, w_pol), dtype=int)
        for r_idx in range(h_pol):
            for c_idx in range(w_pol):
                cell = grid_obj.get(c_idx, r_idx) # grid.get(x,y) -> x=col, y=row
                if cell is None or cell.type != 'wall':
                    map_all_navigable[r_idx, c_idx] = 1 # Mark navigable cell
        print("\nMap with all non-wall states marked as '1':")
        show_map_with_arrows(env_wrapped, map_all_navigable, arrow_type='data', title=f"{current_env_name} Navigable Cells (Marked 1)")
        
    print(f"--- End of Test Sequence for {current_env_name} ---")
    return env_wrapped

print("Helper function run_minigrid_env_test_sequence defined.")

In [ ]:
# Test Custom-LavaCrossing-hard-v0
env_lava_hard_wrapped = run_minigrid_env_test_sequence(
    env_name_custom="Custom-LavaCrossing-hard-v0", 
    fallback_env_name_std="MiniGrid-LavaCrossingS9N2-v0", # S9N2 is harder
    wind_config={'type': 'per_cell', 'strength': 1} # Per-cell wind
)

In [ ]:
# Test Custom-LavaCrossing-extreme-v0
env_lava_extreme_wrapped = run_minigrid_env_test_sequence(
    env_name_custom="Custom-LavaCrossing-extreme-v0", 
    fallback_env_name_std="MiniGrid-LavaCrossingS11N5-v0", # S11N5 is larger and more complex
    wind_config={'type': 'global', 'strength': 2}, # Stronger global wind
    action_to_take=(2, "Forward") # action 2 = Forward
)

In [ ]:
# Test Custom-LavaCrossing-maze-v0
# For fallback, MiniGrid-MazeMultiLava-v0 or MiniGrid-KeyCorridorS3R3-v0 could be options for maze-like
env_lava_maze_wrapped = run_minigrid_env_test_sequence(
    env_name_custom="Custom-LavaCrossing-maze-v0", 
    fallback_env_name_std="MiniGrid-MultiRoom-N4-S5-v0", # A multi-room env as a maze-like fallback
    wind_config={'type': 'per_cell'} # Per-cell wind
)

In [ ]:
# Test Custom-LavaCrossing-maze-complex-v0 with more visualizations
env_lava_maze_complex_wrapped = run_minigrid_env_test_sequence(
    env_name_custom="Custom-LavaCrossing-maze-complex-v0", 
    fallback_env_name_std="MiniGrid-MazeMultiLava-v0", # A complex maze with lava
    wind_config={'type': 'per_cell', 'strength': 1},
    action_to_take=(2, "Forward"), # Take a step forward
    policy_viz_dummy=True,      # Show dummy policy map
    special_viz_requests=True # Trigger all special visualizations for this one
)

## Notebook Completion Notes

This notebook has been generated based on the provided structure.

**Key Placeholders & Assumptions:**

1.  **`WindyWrapper` Mechanics**: The `WindyWrapper` class included is a structural placeholder. It adds wind information (`global_wind_direction` or `wind_map`) and allows manual setting of these. However, **it does not currently implement the physics of how wind would actually affect agent movement within the MiniGrid environment.** This would require modifying the agent's intended position based on wind before MiniGrid's internal collision detection, which is a complex task not implemented here.

2.  **`show_map_with_arrows` Function**: This function is a placeholder for visualizing wind maps, policy maps, or other data overlays on the MiniGrid rendering. Currently, it will print information about the data and show the base environment image but **will not draw actual arrows or overlays.** A full implementation would require graphics routines (e.g., using Matplotlib or Pygame) to draw arrows on each tile according to the data.

3.  **Custom Environment Names**: The notebook attempts to load environments like `"Custom-LavaCrossing-easy-v0"`. If these are not registered in your Gymnasium setup, the code falls back to standard MiniGrid environments and prints an error message.

4.  **Path Adjustments**: The import cell includes logic to adjust `sys.path` to find the `causalgym` module, assuming the notebook is in `causal2/test/` and `causalgym` is in `causal2/causalgym/`. This might need adjustment based on your exact project structure if imports fail.

To make the wind functional and visualizations active, the `WindyWrapper.step()` method and the `show_map_with_arrows` function would need to be further developed.